# Hansen practice: Hansen's RBC model, solved with Uhlig's method

You will solve Hansen's (1985) basic real business cycle model and simulate it:

1. Compute the steady state.
2. Write the log-linearized model as Uhlig's matrices $A, B, C, D, F, \dots, N$.
3. Solve a quadratic for $P$, then find $Q$, $R$, $S$.
4. Plot impulse responses to a TFP shock.
5. Simulate 10,000 quarters and compare the business-cycle moments with the data.

**Model.** The household solves
$$\max\; \mathbb{E}_0\sum_{t=0}^{\infty}\beta^t\bigl(\ln C_t + \psi\ln(1-H_t)\bigr)$$
subject to
$$C_t + K_{t+1} = A_t K_t^{\theta}H_t^{1-\theta} + (1-\delta)K_t, \qquad \ln A_{t+1} = \rho\ln A_t + \varepsilon_{t+1}.$$

Fill in each cell marked `# TODO`, keeping the variable names it asks for. The later cells use them.

In [ ]:
# Run this cell first: it lets your instructor see your progress and error messages during class
import course_reporter; course_reporter.start("hansen_practice")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve

In [ ]:
theta = 0.36      # capital income share
delta = 0.025     # quarterly depreciation
beta  = 0.99      # discount factor
rho   = 0.95      # TFP persistence
sigma = 0.00712   # std. dev. of TFP shock
H_bar = 1/3       # target for steady-state hours (psi is chosen to hit it)

## 1. Steady state

Set $\bar A = 1$.

- **Euler equation:** $1 = \beta(\theta \bar Y/\bar K + 1 - \delta)$ gives $\bar K/\bar Y$.
- **Resource constraint:** $\bar C = \bar Y - \delta\bar K$ gives $\bar C/\bar Y$.
- **Labor–leisure condition:** $\frac{\psi}{1-\bar H} = (1-\theta)\frac{\bar Y}{\bar C\,\bar H}$ gives $\psi$.
- **Levels:** from $\bar Y = \bar K^\theta \bar H^{1-\theta}$, we get $\bar K/\bar H = (\bar K/\bar Y)^{1/(1-\theta)}$.

In [ ]:
# TODO: compute the steady state.
# Name your results: KY (= K/Y), CY (= C/Y), psi, K_bar, Y_bar, C_bar

## 2. Uhlig's matrices

The log-linearized system (tildes are log deviations from the steady state) is

$$
\begin{aligned}
0 &= \mathbb{E}_t\Bigl[\tilde C_t - \tilde C_{t+1} + \beta\theta\tfrac{\bar Y}{\bar K}\bigl(\tilde Y_{t+1} - \tilde K_{t+1}\bigr)\Bigr] && \text{Euler} \\
0 &= \tilde Y_t - \tfrac{1}{1-\bar H}\tilde H_t - \tilde C_t && \text{labor–leisure} \\
0 &= \tilde Y_t - \tilde A_t - \theta\tilde K_t - (1-\theta)\tilde H_t && \text{production} \\
0 &= \bar Y\tilde Y_t - \bar C\tilde C_t + \bar K\bigl[(1-\delta)\tilde K_t - \tilde K_{t+1}\bigr] && \text{resource constraint} \\
\tilde A_{t+1} &= \rho\tilde A_t + \varepsilon_{t+1} && \text{TFP}
\end{aligned}
$$

Uhlig's partition uses the state $x_t = \tilde K_{t+1}$, the other endogenous variables $y_t = [\tilde Y_t, \tilde C_t, \tilde H_t]'$ (in this order), and the exogenous variable $z_t = \tilde A_t$. The system has three blocks:

- **Non-expectational** (rows: labor–leisure, production, resources): $0 = A x_t + B x_{t-1} + C y_t + D z_t$
- **Expectational** (Euler): $0 = \mathbb{E}_t[F x_{t+1} + G x_t + H x_{t-1} + J y_{t+1} + K y_t + L z_{t+1} + M z_t]$
- **Exogenous:** $z_{t+1} = N z_t + \nu_{t+1}$

Use 2-D numpy arrays: $A, B, D$ are $3\times1$, $C$ is $3\times3$, $J$ and $K$ are $1\times3$, and the rest are $1\times1$. Call Uhlig's $H$ and $K$ **`Hm`** and **`Km`**, so they don't clash with hours and capital.

In [ ]:
# TODO: build the matrices A, B, C, D, F, G, Hm, J, Km, L, M, N as 2-D numpy arrays.
# Rows of A, B, C, D: labor-leisure, production, resource constraint.
# Columns of C, J, Km: Y, C, H.

## 3. Solving for $P$

Guess $\tilde K_{t+1} = P\tilde K_t + Q\tilde A_t$ and $y_t = R\tilde K_t + S\tilde A_t$. Matching the coefficients on $\tilde K_t$ gives a quadratic for $P$:

$$(F - JC^{-1}A)\,P^2 - (JC^{-1}B - G + KC^{-1}A)\,P - KC^{-1}B + H = 0.$$

Compute the three coefficients `a2`, `a1`, `a0` as numbers. Use `.item()` to turn a $1\times1$ array into a number. Then find both roots with `np.roots` and keep the **stable** one as `P`.

*Why the stable root?* The other root makes capital explode, which violates the transversality condition.

In [ ]:
# TODO: compute a2, a1, a0, the roots, and the stable root P.
# Print both roots and their product. What should the product equal?

## 4. Solving for $R$, $Q$, $S$

1. $R = -C^{-1}(AP + B)$.
2. $Q = \dfrac{(JC^{-1}D - L)\,N + KC^{-1}D - M}{N(F - JC^{-1}A) + FP + G + JR - KC^{-1}A}$. This formula uses $R$, so compute $R$ first.
3. $S = -C^{-1}(AQ + D)$.

Store `Q` as a number, and `R`, `S` as $3\times1$ arrays.

In [ ]:
# TODO: compute R, Q, S and print the decision rules.

**Check your solution.** Run this cell without changing it. It plugs your decision rules back into all four equations at an arbitrary state, with $\mathbb{E}_t\tilde A_{t+1} = \rho\tilde A_t$. Every residual should be zero.

*Try this:* rerun the cells for $P$ and for $R, Q, S$ with the **explosive** root, then this cell. The residuals are still zero, because both roots satisfy the equations. Only the stability condition tells them apart. Switch back to the stable root before you go on.

In [ ]:
def rules(k, a):
    return P*k + Q*a, R.ravel()*k + S.ravel()*a      # K(t+1), [Y, C, H]

k, a = 0.3, -0.7
k1, (Yt, Ct, Ht) = rules(k, a)
k2, (Y1, C1, H1) = rules(k1, rho*a)
resid = np.array([
    Ct - C1 + beta*theta/KY*(Y1 - k1),                   # Euler
    Yt - Ht/(1 - H_bar) - Ct,                            # labor-leisure
    Yt - a - theta*k - (1 - theta)*Ht,                   # production
    Y_bar*Yt - C_bar*Ct + K_bar*((1 - delta)*k - k1),    # resources
])
print("Residuals:", resid)
assert np.max(np.abs(resid)) < 1e-8, "Your decision rules do not satisfy the model's equations"
print("All equations hold.")

## 5. Impulse responses

Write a function `simulate(A_path)`. It takes a path of $\tilde A_t$, starts from $\tilde K_0 = 0$, iterates the decision rules, and returns a dictionary with the keys `"Y"`, `"C"`, `"H"`, `"K"`, `"I"`, `"A"`. Each entry is an array with the same length as `A_path`, and `"K"` holds $\tilde K_t$ (capital at the start of period $t$).

Investment comes from $I_t = K_{t+1} - (1-\delta)K_t$:
$$\tilde I_t = \frac{\tilde K_{t+1} - (1-\delta)\tilde K_t}{\delta}.$$

Then compute `irf = simulate(0.01 * rho**np.arange(60))`, which is the response to a 1% TFP shock.

In [ ]:
# TODO: write simulate(A_path) and compute irf.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for v in ["Y", "C", "H", "K", "A"]:
    ax[0].plot(100*irf[v], label=v)
ax[1].plot(100*irf["I"], color="k", label="I")
ax[0].set_title("Responses to a 1% TFP shock"); ax[1].set_title("Investment")
for a_ in ax:
    a_.axhline(0, color="gray", lw=0.5); a_.legend()
    a_.set_xlabel("quarters"); a_.set_ylabel("% deviation from steady state")
plt.tight_layout(); plt.show()

**Questions.**

- Why does consumption rise much less than output on impact?
- Why do hours fall *below* the steady state after about 25 quarters?

## 6. Simulating 10,000 quarters

Here is an HP filter ($\lambda = 1600$). It uses sparse matrices, because $T = 10{,}000$ is too large for dense ones. Run the cell without changing it.

In [ ]:
def hp_filter(x, lam=1600):
    """Return the cyclical component of x."""
    T = len(x)
    D2 = sparse.diags([np.ones(T-2), -2*np.ones(T-2), np.ones(T-2)], [0, 1, 2], shape=(T-2, T))
    trend = spsolve((sparse.eye(T) + lam * D2.T @ D2).tocsc(), x)
    return x - trend

Now simulate:

1. Draw $T + 100$ shocks $\varepsilon_t \sim N(0, \sigma^2)$ with `rng = np.random.default_rng(615)`.
2. Build the TFP path $\tilde A_t = \rho\tilde A_{t-1} + \varepsilon_t$, starting from $\tilde A_0 = 0$, and call `simulate`.
3. Drop the first 100 quarters (the burn-in) and HP-filter each series.

Store the filtered series in a dictionary `cyc` with the keys `"Y"`, `"C"`, `"I"`, `"H"`, `"K"`, and add `cyc["Y/H"] = cyc["Y"] - cyc["H"]` for labor productivity.

In [ ]:
# TODO: simulate T = 10,000 quarters (plus a 100-quarter burn-in) and build cyc.

In [ ]:
sdY = cyc["Y"].std()
print(f"{'':6s}{'sd (%)':>8s}{'sd/sd_Y':>9s}{'corr(x,Y)':>11s}{'autocorr':>10s}")
for v, x in cyc.items():
    print(f"{v:6s}{100*x.std():8.2f}{x.std()/sdY:9.2f}"
          f"{np.corrcoef(x, cyc['Y'])[0, 1]:11.2f}{np.corrcoef(x[1:], x[:-1])[0, 1]:10.2f}")

**Compare** your table with U.S. data, where relative volatilities are about 0.73 for consumption, 0.94 for hours and 4.9 for investment, and with Hansen (1985, Table 1).

- Which moments does the model match well?
- Which does it miss? Hint: look at hours.
- How could the model be changed so that hours become more volatile?